In [43]:
import pandas as pd



In [8]:
df_datosSucios = pd.read_csv("mateando-datos.csv")
df_datosSucios.head()

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado
0,L001,P001,01/09/2026,m001,2,"$ 30.000,00",0,MP,PAGADA
1,L002,P001,01/09/2026,B001,2,"7.000,00",0,mercado pago,pagada
2,L003,P002,2026-09-03,M002,1,"80.000,00","5.000,00",Transferencia,pagada
3,L004,P003,05/09/2026,S001,1,"120.000,00","10.000,00",efectivo,pagada
4,L005,P004,08/09/2026,M001,3,"30.000,00","9.000,00",mp,pagada


In [10]:
# 1. Normalizar textos a minúsculas para evitar desajustes
df_datosSucios['medio_pago'] = df_datosSucios['medio_pago'].str.lower().str.strip()
df_datosSucios['estado'] = df_datosSucios['estado'].str.lower().str.strip()

# 2. Estandarizar variantes de pago (ejemplo rápido)
df_datosSucios['medio_pago'] = df_datosSucios['medio_pago'].replace({'mp': 'mercado pago'})

# 3. Ver cómo va quedando el resultado limpio
df_datosSucios.head()

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado
0,L001,P001,01/09/2026,m001,2,"$ 30.000,00",0,mercado pago,pagada
1,L002,P001,01/09/2026,B001,2,"7.000,00",0,mercado pago,pagada
2,L003,P002,2026-09-03,M002,1,"80.000,00","5.000,00",transferencia,pagada
3,L004,P003,05/09/2026,S001,1,"120.000,00","10.000,00",efectivo,pagada
4,L005,P004,08/09/2026,M001,3,"30.000,00","9.000,00",mercado pago,pagada


In [11]:
# 1. Limpiar la columna 'precio_unitario': sacamos el símbolo '$', los puntos de miles, y cambiamos la coma decimal por un punto
df_datosSucios['precio_unitario'] = (
    df_datosSucios['precio_unitario']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .astype(float)
)

# 2. Hacer lo mismo para la columna 'descuento_linea'
df_datosSucios['descuento_linea'] = (
    df_datosSucios['descuento_linea']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .astype(float)
)

# 3. Verificamos cómo quedó la tabla y los tipos de datos
df_datosSucios.head()

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado
0,L001,P001,01/09/2026,m001,2,30000.0,0.0,mercado pago,pagada
1,L002,P001,01/09/2026,B001,2,7000.0,0.0,mercado pago,pagada
2,L003,P002,2026-09-03,M002,1,80000.0,5000.0,transferencia,pagada
3,L004,P003,05/09/2026,S001,1,120000.0,10000.0,efectivo,pagada
4,L005,P004,08/09/2026,M001,3,30000.0,9000.0,mercado pago,pagada


In [15]:
# Convertimos a fecha y luego la formateamos como texto con el formato día/mes/año
df_datosSucios['fecha'] = pd.to_datetime(df_datosSucios['fecha'], format='mixed', dayfirst=True).dt.strftime('%d/%m/%Y')

# Verificamos la tabla resultante
df_datosSucios.head()

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado
0,L001,P001,01/09/2026,m001,2,30000.0,0.0,mercado pago,pagada
1,L002,P001,01/09/2026,B001,2,7000.0,0.0,mercado pago,pagada
2,L003,P002,09/03/2026,M002,1,80000.0,5000.0,transferencia,pagada
3,L004,P003,05/09/2026,S001,1,120000.0,10000.0,efectivo,pagada
4,L005,P004,08/09/2026,M001,3,30000.0,9000.0,mercado pago,pagada


In [20]:
# Calcular la facturación total (cantidad * precio_unitario) agrupada por medio de pago
df_datosSucios['total_venta'] = df_datosSucios['cantidad'] * df_datosSucios['precio_unitario']

ventas_por_pago = df_datosSucios.groupby('medio_pago')['total_venta'].sum().reset_index()
ventas_por_pago.head()

,medio_pago,total_venta
0,efectivo,475000.0
1,mercado pago,550000.0
2,transferencia,320000.0


In [21]:
# Agrupar por SKU (producto) sumando las cantidades y los ingresos totales
productos_mas_vendidos = df_datosSucios.groupby('sku').agg({
    'cantidad': 'sum',
    'total_venta': 'sum'
}).reset_index().sort_values(by='total_venta', ascending=False)

productos_mas_vendidos

,sku,cantidad,total_venta
3,S001,4,480000.0
1,M001,11,330000.0
2,M002,4,320000.0
0,B001,15,105000.0
5,m001,2,60000.0
4,X999,1,50000.0


In [22]:
# Agrupar por fecha sumando la facturación total
ventas_por_fecha = df_datosSucios.groupby('fecha')['total_venta'].sum().reset_index()
ventas_por_fecha

,fecha,total_venta
0,01/09/2026,74000.0
1,05/09/2026,120000.0
2,08/09/2026,180000.0
3,09/03/2026,80000.0
4,10/09/2026,35000.0
5,12/09/2026,160000.0
6,15/09/2026,240000.0
7,18/09/2026,120000.0
8,20/09/2026,30000.0
9,21/09/2026,56000.0


In [ ]:
# Guardar el DataFrame limpio en un nuevo CSV
df_datosSucios.to_csv("mateando_datos_limpios.csv", index=False)

In [24]:
# Te muestra los tipos de datos de cada columna
df_datosSucios.dtypes

id_linea               str
pedido                 str
fecha                  str
sku                    str
cantidad             int64
precio_unitario    float64
descuento_linea    float64
medio_pago             str
estado                 str
total_linea        float64
total_venta        float64
dtype: object

In [25]:
# Eliminamos la columna redundante 'total_linea'
df_datosSucios = df_datosSucios.drop(columns=['total_linea'])

In [26]:
# Muestra el DataFrame completo en una tabla interactiva y prolija
display(df_datosSucios)

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado,total_venta
0,L001,P001,01/09/2026,m001,2,30000.0,0.0,mercado pago,pagada,60000.0
1,L002,P001,01/09/2026,B001,2,7000.0,0.0,mercado pago,pagada,14000.0
2,L003,P002,09/03/2026,M002,1,80000.0,5000.0,transferencia,pagada,80000.0
3,L004,P003,05/09/2026,S001,1,120000.0,10000.0,efectivo,pagada,120000.0
4,L005,P004,08/09/2026,M001,3,30000.0,9000.0,mercado pago,pagada,90000.0
5,L006,P005,10/09/2026,B001,5,7000.0,0.0,efectivo,pagada,35000.0
6,L007,P006,12/09/2026,M002,2,80000.0,0.0,transferencia,pagada,160000.0
7,L008,P007,15/09/2026,S001,2,120000.0,20000.0,mercado pago,pagada,240000.0
8,L009,P008,18/09/2026,M001,4,30000.0,0.0,efectivo,pagada,120000.0
9,L010,P009,21/09/2026,B001,8,7000.0,8000.0,mercado pago,pagada,56000.0


In [27]:
# Limpiamos espacios y capitalizamos 'medio_pago' (ej: "mercado pago" -> "Mercado pago")
df_datosSucios['medio_pago'] = df_datosSucios['medio_pago'].astype(str).str.strip().str.capitalize()

# Hacemos lo mismo con 'estado'
df_datosSucios['estado'] = df_datosSucios['estado'].astype(str).str.strip().str.capitalize()

In [28]:
display(df_datosSucios)

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado,total_venta
0,L001,P001,01/09/2026,m001,2,30000.0,0.0,Mercado pago,Pagada,60000.0
1,L002,P001,01/09/2026,B001,2,7000.0,0.0,Mercado pago,Pagada,14000.0
2,L003,P002,09/03/2026,M002,1,80000.0,5000.0,Transferencia,Pagada,80000.0
3,L004,P003,05/09/2026,S001,1,120000.0,10000.0,Efectivo,Pagada,120000.0
4,L005,P004,08/09/2026,M001,3,30000.0,9000.0,Mercado pago,Pagada,90000.0
5,L006,P005,10/09/2026,B001,5,7000.0,0.0,Efectivo,Pagada,35000.0
6,L007,P006,12/09/2026,M002,2,80000.0,0.0,Transferencia,Pagada,160000.0
7,L008,P007,15/09/2026,S001,2,120000.0,20000.0,Mercado pago,Pagada,240000.0
8,L009,P008,18/09/2026,M001,4,30000.0,0.0,Efectivo,Pagada,120000.0
9,L010,P009,21/09/2026,B001,8,7000.0,8000.0,Mercado pago,Pagada,56000.0


In [29]:
# Convertimos todos los SKU a mayúsculas y quitamos espacios sobrantes
df_datosSucios['sku'] = df_datosSucios['sku'].astype(str).str.strip().str.upper()

In [30]:
display(df_datosSucios)

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado,total_venta
0,L001,P001,01/09/2026,M001,2,30000.0,0.0,Mercado pago,Pagada,60000.0
1,L002,P001,01/09/2026,B001,2,7000.0,0.0,Mercado pago,Pagada,14000.0
2,L003,P002,09/03/2026,M002,1,80000.0,5000.0,Transferencia,Pagada,80000.0
3,L004,P003,05/09/2026,S001,1,120000.0,10000.0,Efectivo,Pagada,120000.0
4,L005,P004,08/09/2026,M001,3,30000.0,9000.0,Mercado pago,Pagada,90000.0
5,L006,P005,10/09/2026,B001,5,7000.0,0.0,Efectivo,Pagada,35000.0
6,L007,P006,12/09/2026,M002,2,80000.0,0.0,Transferencia,Pagada,160000.0
7,L008,P007,15/09/2026,S001,2,120000.0,20000.0,Mercado pago,Pagada,240000.0
8,L009,P008,18/09/2026,M001,4,30000.0,0.0,Efectivo,Pagada,120000.0
9,L010,P009,21/09/2026,B001,8,7000.0,8000.0,Mercado pago,Pagada,56000.0


In [34]:
from sqlalchemy import create_engine

# Creamos el motor de la base de datos local
engine = create_engine('sqlite:///mateando_datos.db')

# Volcamos el DataFrame limpio a la tabla 'ventas'
df_datosSucios.to_sql('ventas', con=engine, if_exists='replace', index=False)

print("¡Base de datos generada con éxito: mateando_datos.db!")

¡Base de datos generada con éxito: mateando_datos.db!


In [35]:
# Convertimos a datetime y luego a string en formato YYYY-MM-DD
df_datosSucios['fecha'] = pd.to_datetime(
    df_datosSucios['fecha'], format='%d/%m/%Y'
).dt.strftime('%Y-%m-%d')

# Vuelves a correr tu .to_sql para pisar la tabla limpia
df_datosSucios.to_sql('ventas', con=engine, if_exists='replace', index=False)

15

In [36]:
display(df_datosSucios)

,id_linea,pedido,fecha,sku,cantidad,precio_unitario,descuento_linea,medio_pago,estado,total_venta
0,L001,P001,2026-09-01,M001,2,30000.0,0.0,Mercado pago,Pagada,60000.0
1,L002,P001,2026-09-01,B001,2,7000.0,0.0,Mercado pago,Pagada,14000.0
2,L003,P002,2026-03-09,M002,1,80000.0,5000.0,Transferencia,Pagada,80000.0
3,L004,P003,2026-09-05,S001,1,120000.0,10000.0,Efectivo,Pagada,120000.0
4,L005,P004,2026-09-08,M001,3,30000.0,9000.0,Mercado pago,Pagada,90000.0
5,L006,P005,2026-09-10,B001,5,7000.0,0.0,Efectivo,Pagada,35000.0
6,L007,P006,2026-09-12,M002,2,80000.0,0.0,Transferencia,Pagada,160000.0
7,L008,P007,2026-09-15,S001,2,120000.0,20000.0,Mercado pago,Pagada,240000.0
8,L009,P008,2026-09-18,M001,4,30000.0,0.0,Efectivo,Pagada,120000.0
9,L010,P009,2026-09-21,B001,8,7000.0,8000.0,Mercado pago,Pagada,56000.0


In [37]:
from sqlalchemy import create_engine

# Creamos el motor de la base de datos local
engine = create_engine('sqlite:///mateando_datos.db')

# Volcamos el DataFrame limpio a la tabla 'ventas'
df_datosSucios.to_sql('ventas', con=engine, if_exists='replace', index=False)

print("¡Base de datos generada con éxito: mateando_datos.db!")

¡Base de datos generada con éxito: mateando_datos.db!


In [40]:
import pandas as pd
from sqlalchemy import create_engine

# Nos conectamos a la base de datos
engine = create_engine('sqlite:///mateando_datos.db')

# 1. Creamos la tabla de ventas diarias
query_diarias = """
    CREATE TABLE IF NOT EXISTS dash_ventas_diarias AS
    SELECT 
        fecha,
        COUNT(DISTINCT pedido) AS total_pedidos,
        SUM(cantidad) AS total_unidades_vendidas,
        SUM(total_venta) AS facturacion_total,
        AVG(total_venta) AS ticket_promedio
    FROM ventas
    WHERE estado = 'Pagada'
    GROUP BY fecha
    ORDER BY fecha ASC;
"""

# 2. Creamos la tabla del ranking de productos
query_ranking = """
    CREATE TABLE IF NOT EXISTS ranking_productos AS
    SELECT sku, SUM(cantidad) AS total_unidades, SUM(total_venta) AS facturacion_total
    FROM ventas
    WHERE estado = 'Pagada'
    GROUP BY sku
    ORDER BY total_unidades DESC;
"""

# 3. Creamos la tabla de resumen de estados
query_estados = """
    CREATE TABLE IF NOT EXISTS resumen_estados AS
    SELECT estado, COUNT(DISTINCT pedido) AS total_pedidos, SUM(total_venta) AS dinero_involucrado
    FROM ventas
    GROUP BY estado;
"""

# Ejecutamos las creaciones directo en la base de datos
with engine.connect() as conexion:
    conexion.exec_driver_sql(query_diarias)
    conexion.exec_driver_sql(query_ranking)
    conexion.exec_driver_sql(query_estados)
    conexion.commit()

print("¡Tablas creadas y guardadas con éxito en la base de datos!")

¡Tablas creadas y guardadas con éxito en la base de datos!


In [42]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Nos conectamos a nuestra base de datos local
engine = create_engine('sqlite:///mateando_datos.db')

# 2. Leemos cada una de las tablas que creamos en SQLite y las metemos en DataFrames
df_ventas_diarias = pd.read_sql("SELECT * FROM dash_ventas_diarias", con=engine)
df_ranking = pd.read_sql("SELECT * FROM ranking_productos", con=engine)
df_estados = pd.read_sql("SELECT * FROM resumen_estados", con=engine)

# 3. Creamos un archivo Excel y guardamos cada DataFrame en una pestaña distinta
nombre_archivo = "reporte_mateando_datos.xlsx"

with pd.ExcelWriter(nombre_archivo, engine='openpyxl') as writer:
    df_ventas_diarias.to_excel(writer, sheet_name='Ventas Diarias', index=False)
    df_ranking.to_excel(writer, sheet_name='Ranking Productos', index=False)
    df_estados.to_excel(writer, sheet_name='Resumen Estados', index=False)

print(f"¡Listo! Archivo generado con éxito: {nombre_archivo}")

¡Listo! Archivo generado con éxito: reporte_mateando_datos.xlsx


## Desafío de SQL Avanzado: Relaciones y JOINS con Clientes

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Nos conectamos a la base de datos existente
engine = create_engine('sqlite:///mateando_datos.db')

# 2. Creamos la tabla de Clientes
query_crear_clientes = """
    CREATE TABLE IF NOT EXISTS clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre_cliente TEXT,
        email TEXT,
        provincia TEXT
    );
"""

# 3. Insertamos datos de ejemplo en la tabla clientes
query_insertar_clientes = """
    INSERT OR IGNORE INTO clientes (id_cliente, nombre_cliente, email, provincia) VALUES
    (1, 'Lucía Gómez', 'luciaroman@email.com', 'Buenos Aires'),
    (2, 'Carlos Pérez', 'carlosjuarez@email.com', 'Córdoba'),
    (3, 'Mariana Benítez', 'marianagomez@email.com', 'Santa Fe'),
    (4, 'Federico Ruiz', 'fedemamani@email.com', 'Mendoza'),
    (5, 'Sofía Martínez', 'sofiamedina@email.com', 'Buenos Aires');
"""

# Ejecutamos la creación e inserción en SQLite
with engine.connect() as conexion:
    conexion.exec_driver_sql(query_crear_clientes)
    conexion.exec_driver_sql(query_insertar_clientes)
    conexion.commit()

print("¡Tabla de clientes creada e inicializada con éxito en la base de datos!")

¡Tabla de clientes creada e inicializada con éxito en la base de datos!


In [47]:
# ==========================================
# AGREGAMOS ID_CLIENTE A LA TABLA VENTAS
# ==========================================

# 1. Agregamos la columna id_cliente a la tabla ventas existente
query_agregar_columna = """
    ALTER TABLE ventas ADD COLUMN id_cliente INTEGER;
"""

# 2. Actualizamos los registros de ventas asignando un id_cliente al azar (del 1 al 5)
query_actualizar_ventas = """
    UPDATE ventas 
    SET id_cliente = (ABS(RANDOM()) % 5) + 1;
"""

with engine.connect() as conexion:
    try:
        conexion.exec_driver_sql(query_agregar_columna)
    except Exception as e:
        print("La columna id_cliente ya existía, continuando...")
        
    conexion.exec_driver_sql(query_actualizar_ventas)
    conexion.commit()

print("¡Tabla ventas actualizada con éxito con los ID de clientes!")

¡Tabla ventas actualizada con éxito con los ID de clientes!


In [48]:


query_join_clientes = """
    SELECT 
        c.nombre_cliente,
        c.provincia,
        COUNT(v.pedido) AS total_pedidos,
        SUM(v.total_venta) AS facturacion_total
    FROM ventas v
    JOIN clientes c ON v.id_cliente = c.id_cliente
    WHERE v.estado = 'Pagada'
    GROUP BY c.nombre_cliente, c.provincia
    ORDER BY facturacion_total DESC;
"""

# Leemos el resultado directamente en un DataFrame de Pandas
df_clientes_top = pd.read_sql(query_join_clientes, con=engine)

# Mostramos el resultado por pantalla
print("=== TOP CLIENTES SEGÚN FACTURACIÓN ===")
display(df_clientes_top)

=== TOP CLIENTES SEGÚN FACTURACIÓN ===


,nombre_cliente,provincia,total_pedidos,facturacion_total
0,Federico Ruiz,Mendoza,4,530000.0
1,Sofía Martínez,Buenos Aires,3,370000.0
2,Mariana Benítez,Santa Fe,2,180000.0
3,Carlos Pérez,Córdoba,3,165000.0
4,Lucía Gómez,Buenos Aires,2,70000.0


In [49]:
# 1. Insertamos un cliente nuevo que NO tiene compras asociadas
query_nuevo_cliente = """
    INSERT OR IGNORE INTO clientes (id_cliente, nombre_cliente, email, provincia) 
    VALUES (6, 'Martina López', 'martina@email.com', 'Tucumán');
"""

In [50]:
# 2. Ejecutamos un LEFT JOIN desde la tabla clientes hacia ventas
# Esto nos mostrará a TODOS los clientes, incluso si sus ventas son 0 o NULL
query_left_join = """
    SELECT 
        c.id_cliente,
        c.nombre_cliente,
        c.provincia,
        COUNT(v.pedido) AS total_pedidos,
        COALESCE(SUM(v.total_venta), 0) AS facturacion_total
    FROM clientes c
    LEFT JOIN ventas v ON c.id_cliente = v.id_cliente
    GROUP BY c.id_cliente, c.nombre_cliente, c.provincia
    ORDER BY facturacion_total DESC;
"""

with engine.connect() as conexion:
    conexion.exec_driver_sql(query_nuevo_cliente)
    conexion.commit()

# Leemos el resultado con Pandas
df_left_join = pd.read_sql(query_left_join, con=engine)

print("=== RESULTADO DE LEFT JOIN (TODOS LOS CLIENTES) ===")
display(df_left_join)

=== RESULTADO DE LEFT JOIN (TODOS LOS CLIENTES) ===


,id_cliente,nombre_cliente,provincia,total_pedidos,facturacion_total
0,4,Federico Ruiz,Mendoza,4,530000.0
1,5,Sofía Martínez,Buenos Aires,3,370000.0
2,3,Mariana Benítez,Santa Fe,3,210000.0
3,2,Carlos Pérez,Córdoba,3,165000.0
4,1,Lucía Gómez,Buenos Aires,2,70000.0
5,6,Martina López,Tucumán,0,0.0


In [51]:
# 1. Creamos la tabla metodos_pago
query_crear_pagos = """
    CREATE TABLE IF NOT EXISTS metodos_pago (
        id_metodo INTEGER PRIMARY KEY,
        nombre_metodo TEXT
    );
"""

# 2. Insertamos los datos de los medios de pago
query_insertar_pagos = """
    INSERT OR IGNORE INTO metodos_pago (id_metodo, nombre_metodo) VALUES
    (1, 'Mercado Pago'),
    (2, 'Tarjeta de Crédito'),
    (3, 'Transferencia Bancaria'),
    (4, 'Efectivo');
"""

with engine.connect() as conexion:
    conexion.exec_driver_sql(query_crear_pagos)
    conexion.exec_driver_sql(query_insertar_pagos)
    conexion.commit()

print("¡Tabla de métodos de pago creada e inicializada con éxito!")

¡Tabla de métodos de pago creada e inicializada con éxito!


In [52]:
# ==========================================
# VINCULAR METODOS DE PAGO A LAS VENTAS
# ==========================================

# 1. Agregamos la columna id_metodo a la tabla ventas
query_agregar_metodo = """
    ALTER TABLE ventas ADD COLUMN id_metodo INTEGER;
"""

# 2. Asignamos un método de pago aleatorio (del 1 al 4) a cada venta
query_actualizar_pagos_ventas = """
    UPDATE ventas 
    SET id_metodo = (ABS(RANDOM()) % 4) + 1;
"""

with engine.connect() as conexion:
    try:
        conexion.exec_driver_sql(query_agregar_metodo)
    except Exception as e:
        print("La columna id_metodo ya existía, continuando...")
        
    conexion.exec_driver_sql(query_actualizar_pagos_ventas)
    conexion.commit()

print("¡Tabla ventas actualizada con los métodos de pago!")

¡Tabla ventas actualizada con los métodos de pago!


In [53]:
# ==========================================
# CONSULTA  para ver un reporte de quién compró, qué pagó y con qué medio de pago lo hizo
# ==========================================

query_doble_join = """
    SELECT 
        c.nombre_cliente,
        mp.nombre_metodo AS metodo_pago,
        COUNT(v.pedido) AS total_pedidos,
        SUM(v.total_venta) AS facturacion_total
    FROM ventas v
    JOIN clientes c ON v.id_cliente = c.id_cliente
    JOIN metodos_pago mp ON v.id_metodo = mp.id_metodo
    WHERE v.estado = 'Pagada'
    GROUP BY c.nombre_cliente, mp.nombre_metodo
    ORDER BY facturacion_total DESC;
"""

df_reporte_completo = pd.read_sql(query_doble_join, con=engine)

print("=== REPORTE COMPLETO: CLIENTES Y MEDIOS DE PAGO ===")
display(df_reporte_completo)

=== REPORTE COMPLETO: CLIENTES Y MEDIOS DE PAGO ===


,nombre_cliente,metodo_pago,total_pedidos,facturacion_total
0,Federico Ruiz,Mercado Pago,1,240000.0
1,Carlos Pérez,Transferencia Bancaria,3,165000.0
2,Sofía Martínez,Efectivo,1,160000.0
3,Federico Ruiz,Tarjeta de Crédito,1,120000.0
4,Mariana Benítez,Mercado Pago,1,120000.0
5,Sofía Martínez,Transferencia Bancaria,1,120000.0
6,Federico Ruiz,Transferencia Bancaria,1,90000.0
7,Sofía Martínez,Tarjeta de Crédito,1,90000.0
8,Federico Ruiz,Efectivo,1,80000.0
9,Mariana Benítez,Transferencia Bancaria,1,60000.0
